In [32]:
import pandas as pd
import openpyxl
import ifcopenshell
import ifcopenshell.util.element
from collections import Counter


In [33]:
df_matrix = pd.read_excel(r'C:\Users\lucas.galicioli\ifc-classifier\data\interim\classification-matrix.xlsx')
display(df_matrix.head())

,Class,PredefinedType,BuildingStorey,Material,Name,PSET_RÔGGA.RÔGGA_SEÇÃO,PSET_RÔGGA.RÔGGA_DESCRIÇÃO,Width,Thickness,Length,Height,FileName,Ô_CLS_DISCIPLINAS,Ô_CLS_CLASSIFICAÇÃO_SOLIBRI
0,IfcFlowSegment,NOTDEFINED,01. TCO,NaN,Tipos de tubos:Tubo - AÇO Hidrante/Sprinkler:1...,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-SPK-EX-0001-BIM-EMB-GER-SPRINKLERS-R01,Preventivo contra incêndio - SPK,Tubulação
1,IfcFlowSegment,NOTDEFINED,01.1. TER,NaN,Tipos de tubos:Tubo - AÇO Hidrante/Sprinkler:1...,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-SPK-EX-0001-BIM-EMB-GER-SPRINKLERS-R01,Preventivo contra incêndio - SPK,Tubulação
2,IfcFlowSegment,NOTDEFINED,01. TCO,NaN,Tipos de tubos:Tubo - AÇO Hidrante/Sprinkler:1...,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-SPK-EX-0001-BIM-EMB-GER-SPRINKLERS-R01,Preventivo contra incêndio - SPK,Tubulação
3,IfcFlowSegment,NOTDEFINED,01.1. TER,NaN,Tipos de tubos:Tubo - AÇO Hidrante/Sprinkler:1...,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-SPK-EX-0001-BIM-EMB-GER-SPRINKLERS-R01,Preventivo contra incêndio - SPK,Tubulação
4,IfcFlowSegment,NOTDEFINED,01. TCO,NaN,Tipos de tubos:Tubo - AÇO Hidrante/Sprinkler:1...,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-SPK-EX-0001-BIM-EMB-GER-SPRINKLERS-R01,Preventivo contra incêndio - SPK,Tubulação


In [34]:
df_matrix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148128 entries, 0 to 148127
Data columns (total 14 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Class                        148128 non-null  object 
 1   PredefinedType               139027 non-null  object 
 2   BuildingStorey               148128 non-null  object 
 3   Material                     105558 non-null  object 
 4   Name                         147755 non-null  object 
 5   PSET_RÔGGA.RÔGGA_SEÇÃO       59244 non-null   object 
 6   PSET_RÔGGA.RÔGGA_DESCRIÇÃO   116827 non-null  object 
 7   Width                        3063 non-null    float64
 8   Thickness                    37310 non-null   float64
 9   Length                       25837 non-null   float64
 10  Height                       30750 non-null   float64
 11  FileName                     148128 non-null  object 
 12  Ô_CLS_DISCIPLINAS            148128 non-null  object 
 13 

In [35]:
df_matrix.describe()

,Width,Thickness,Length,Height
count,3063.000000,37310.000000,25837.000000,30750.000000
mean,0.341347,0.037275,0.721532,1.656536
std,0.618977,0.275060,1.841621,2.707004
min,0.000000,0.000000,0.000000,0.010000
25%,0.000000,0.000000,0.100000,0.446000
50%,0.000000,0.001100,0.200000,0.950000
75%,0.660000,0.002000,0.940000,1.940200
max,7.240000,21.229900,101.125500,86.902400


In [36]:
df_matrix.isnull().sum()

Class                               0
PredefinedType                   9101
BuildingStorey                      0
Material                        42570
Name                              373
PSET_RÔGGA.RÔGGA_SEÇÃO          88884
PSET_RÔGGA.RÔGGA_DESCRIÇÃO      31301
Width                          145065
Thickness                      110818
Length                         122291
Height                         117378
FileName                            0
Ô_CLS_DISCIPLINAS                   0
Ô_CLS_CLASSIFICAÇÃO_SOLIBRI         0
dtype: int64

In [37]:
import numpy as np # Importante ter o numpy

# Lista das colunas de texto que queremos limpar
colunas_para_limpar = [
    'Material',
    'PSET_RÔGGA.RÔGGA_SEÇÃO',
    'PSET_RÔGGA.RÔGGA_DESCRIÇÃO'

]

# Valor que representa "ausente" no seu arquivo
valor_ausente = 'NaN'

# Novo valor que vamos colocar no lugar
novo_valor = 'Desconhecido'

# Este loop vai passar por cada coluna da lista e fazer a substituição
for coluna in colunas_para_limpar:
    print(f"Limpando a coluna: {coluna}...")
    df_matrix[coluna] = df_matrix[coluna].replace(valor_ausente, novo_valor)

print("\nLimpeza concluída!")

Limpando a coluna: Material...
Limpando a coluna: PSET_RÔGGA.RÔGGA_SEÇÃO...
Limpando a coluna: PSET_RÔGGA.RÔGGA_DESCRIÇÃO...

Limpeza concluída!


In [38]:
display(df_matrix['Material'].value_counts())

Material
Concreto C40                                          9988
Linha Corrugado PVC Reforçado Laranja Antichamas      8761
Tela de Poliester                                     7652
<Unnamed>                                             5201
PVC Antichama Branco Elétrica                         4931
                                                      ... 
Metal-Garaventa Lift-Aluminium;<Unnamed>                 1
Peinture - Blanc;Plastique - Bleu;Plastique - Noir       1
Plastic - Black;Metal - Crome                            1
Piso Vinílico Gergelim - Ambienta Tarkett                1
wire_127127127;wire_255255255;wire_025025025             1
Name: count, Length: 400, dtype: int64

In [39]:
# Seleciona as colunas de entrada (features) para o modelo
features_selecionadas = [
    'Class',
    'PredefinedType',
    'BuildingStorey',
    'Material',
    'PSET_RÔGGA.RÔGGA_SEÇÃO',
    'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
    'Width',
    'Thickness',
    'Length',
    'Height'
]

X = df_matrix[features_selecionadas]

print("DataFrame 'X' criado com as features selecionadas.")

DataFrame 'X' criado com as features selecionadas.


In [40]:
# Aplica o One-Hot Encoding em todas as colunas de texto dentro de X
X_encoded = pd.get_dummies(X)

print("One-Hot Encoding concluído!")

One-Hot Encoding concluído!


In [41]:
# 1. Mostra as dimensões (linhas, colunas) do novo DataFrame.
# O número de colunas vai aumentar BASTANTE!
print(X_encoded.shape)

# 2. Mostra as 5 primeiras linhas do novo DataFrame para vermos a estrutura
print(X_encoded.head())

(148128, 1761)
   Width  Thickness  Length  Height  Class_IfcAirTerminal  Class_IfcAlarm  \
0    NaN        NaN     NaN     NaN                 False           False   
1    NaN        NaN     NaN     NaN                 False           False   
2    NaN        NaN     NaN     NaN                 False           False   
3    NaN        NaN     NaN     NaN                 False           False   
4    NaN        NaN     NaN     NaN                 False           False   

   Class_IfcBeam  Class_IfcBuildingElementProxy  Class_IfcCableCarrierFitting  \
0          False                          False                         False   
1          False                          False                         False   
2          False                          False                         False   
3          False                          False                         False   
4          False                          False                         False   

   Class_IfcCableCarrierSegment  ..

In [42]:
# Lista das colunas numéricas
colunas_numericas = ['Width', 'Thickness', 'Length', 'Height']

# Este loop passa por cada coluna da lista
for coluna in colunas_numericas:
    # 1. Calcula a mediana da coluna
    mediana = X_encoded[coluna].median()
    
    # 2. Usa .fillna() para preencher os NaN com o valor da mediana
    X_encoded[coluna] = X_encoded[coluna].fillna(mediana)
    
    print(f"Valores nulos na coluna '{coluna}' preenchidos com a mediana ({mediana}).")

print("\nPreenchimento de dados numéricos concluído!")

Valores nulos na coluna 'Width' preenchidos com a mediana (0.0).
Valores nulos na coluna 'Thickness' preenchidos com a mediana (0.0011).
Valores nulos na coluna 'Length' preenchidos com a mediana (0.2).
Valores nulos na coluna 'Height' preenchidos com a mediana (0.95).

Preenchimento de dados numéricos concluído!


In [43]:
# Soma todos os valores nulos em todas as colunas do DataFrame
total_nulos = X_encoded.isnull().sum().sum()

print(f"\nTotal de valores nulos no DataFrame final 'X_encoded': {total_nulos}")

if total_nulos == 0:
    print("Parabéns! Seus dados estão 100% limpos e prontos para o treinamento do modelo!")
else:
    print("Ainda existem valores nulos. Precisamos investigar o que aconteceu.")


Total de valores nulos no DataFrame final 'X_encoded': 0
Parabéns! Seus dados estão 100% limpos e prontos para o treinamento do modelo!


In [44]:
from sklearn.preprocessing import LabelEncoder

# 1. Seleciona a primeira coluna que queremos prever do DataFrame original
y1 = df_matrix['Ô_CLS_DISCIPLINAS']

# 2. Cria uma instância do codificador
le1 = LabelEncoder()

# 3.Ajusta o codificador aos seus dados e os transforma em números
y1_encoded = le1.fit_transform(y1)

print("Variável alvo 'Ô_CLS_DISCIPLINAS' foi codificada com sucesso!")
print("Exemplo dos primeiros 5 valores codificados:", y1_encoded[:5])


Variável alvo 'Ô_CLS_DISCIPLINAS' foi codificada com sucesso!
Exemplo dos primeiros 5 valores codificados: [14 14 14 14 14]


In [45]:
from sklearn.model_selection import train_test_split

# ==============================================================================
# BLOCO DE CÓDIGO CONSOLIDADO - SUBSTITUA SUAS CÉLULAS ANTERIORES POR ESTE
# ==============================================================================

print("Passo 1 de 3: Aplicando One-Hot Encoding...")
X_encoded = pd.get_dummies(X)
print(f"--> DataFrame encodado criado com shape: {X_encoded.shape}")

# ------------------------------------------------------------------------------

print("\nPasso 2 de 3: Limpando nomes das colunas para compatibilidade com XGBoost...")
# Esta linha agora remove '[', ']', e '<'
X_encoded.columns = X_encoded.columns.str.replace(r'\[|\]|<', '_', regex=True)
print("--> Limpeza dos nomes das colunas concluída!")

# ------------------------------------------------------------------------------

print("\nPasso 3 de 3: Dividindo os dados em conjuntos de treino e teste...")
# Este passo AGORA usa o X_encoded COM os nomes já limpos.
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y1_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y1_encoded
)
print("--> Dados divididos com sucesso!")
print(f"--> Shape do X_train final: {X_train.shape}")
print("\n✅ PREPARAÇÃO CONCLUÍDA. Agora você pode rodar a célula do GridSearchCV com segurança.")
# ==============================================================================

Passo 1 de 3: Aplicando One-Hot Encoding...
--> DataFrame encodado criado com shape: (148128, 1761)

Passo 2 de 3: Limpando nomes das colunas para compatibilidade com XGBoost...
--> Limpeza dos nomes das colunas concluída!

Passo 3 de 3: Dividindo os dados em conjuntos de treino e teste...
--> Dados divididos com sucesso!
--> Shape do X_train final: (111096, 1761)

✅ PREPARAÇÃO CONCLUÍDA. Agora você pode rodar a célula do GridSearchCV com segurança.


In [46]:
# ==============================================================================
# BLOCO DE CÓDIGO CORRIGIDO PARA TREINAMENTO COM KNN
# ==============================================================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer  # <--- Importe o Imputer
from sklearn.metrics import accuracy_score, classification_report

# --- 1. Escalar os Dados (Como antes) ---
print("Iniciando o escalonamento dos dados...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("-> Dados de treino e teste escalonados.")

# --- 2. Imputar NaNs Gerados (NOVO PASSO DE CORREÇÃO) ---
print("\nLimpando NaNs gerados pelo escalonamento...")
#    Vamos substituir qualquer NaN pela média da sua coluna (que é 0 após o scaling)
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train_scaled)
X_test_imputed = imputer.transform(X_test_scaled)
print("-> Limpeza concluída! Os dados estão prontos para o KNN.")

# --- 3. Treinar o Modelo KNN (Agora com dados 100% limpos) ---
print("\nIniciando o treinamento do modelo KNN...")
knn_classifier = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

# Treina o modelo com os dados de treino escalados E IMPUTADOS
knn_classifier.fit(X_train_imputed, y_train)

print("-> Treinamento do KNN concluído!")

# --- 4. Avaliar o Modelo KNN ---
print("\nRealizando previsões com o modelo KNN...")
# Usa os dados de teste escalados E IMPUTADOS para a previsão
y_pred_knn = knn_classifier.predict(X_test_imputed)

# Avalia a acurácia
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"\nAcurácia do modelo KNN: {accuracy_knn:.4f}")
print(f"Isso significa que o modelo KNN acertou {accuracy_knn:.2%} das classificações!")

# Gera o relatório de classificação detalhado
target_names = le1.classes_
print("\nRelatório de Classificação Detalhado (KNN):")
print(classification_report(y_test, y_pred_knn, target_names=target_names, zero_division=0))
# ==============================================================================

Iniciando o escalonamento dos dados...
-> Dados de treino e teste escalonados.

Limpando NaNs gerados pelo escalonamento...
-> Limpeza concluída! Os dados estão prontos para o KNN.

Iniciando o treinamento do modelo KNN...
-> Treinamento do KNN concluído!

Realizando previsões com o modelo KNN...

Acurácia do modelo KNN: 0.9827
Isso significa que o modelo KNN acertou 98.27% das classificações!

Relatório de Classificação Detalhado (KNN):
                                  precision    recall  f1-score   support

                     Arquitetura       0.99      0.99      0.99      4559
                    Climatização       1.00      1.00      1.00      1936
                        Elétrica       0.99      0.98      0.99      8565
                       Estrutura       1.00      1.00      1.00      2497
                             Gás       1.00      1.00      1.00      1973
                      Hidráulica       0.99      0.97      0.98      4618
               Impermeabilização       

In [51]:
import joblib

# Salva o modelo treinado em um arquivo
joblib.dump(knn_classifier, 'ifc_classifier_disciplinas_v1.pkl')

# Salva também o LabelEncoder, pois você precisará dele para decodificar as previsões
joblib.dump(le1, 'label_encoder_disciplinas_v1.pkl')

print("Modelo e LabelEncoder salvos com sucesso!")

Modelo e LabelEncoder salvos com sucesso!


In [52]:
import json

# --- SALVAR ARTEFATOS ADICIONAIS ---

# 3. Salvar a lista de colunas do modelo
colunas_do_modelo = X_encoded.columns.tolist()
with open('colunas_modelo_disciplinas.json', 'w') as f:
    json.dump(colunas_do_modelo, f)

# 4. Salvar as medianas de treinamento
colunas_numericas = ['Width', 'Thickness', 'Length', 'Height']
medianas = df_matrix[colunas_numericas].median().to_dict()
with open('medianas_treinamento.json', 'w') as f:
    json.dump(medianas, f)

print("Artefatos de pré-processamento (colunas e medianas) salvos com sucesso!")

Artefatos de pré-processamento (colunas e medianas) salvos com sucesso!


In [ ]:
# import ifcopenshell
# import ifcopenshell.util.element
# import pandas as pd
# import os

# # --- Caminho do arquivo IFC ---
# ifc_file_path = r'C:\Users\lucas.galicioli\ifc-classifier\data\raw\PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00.ifc'

# # --- Funções Auxiliares (com a função de quantidade modificada) ---

# def get_building_storey(element):
#     """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
#     try:
#         spatial_container = ifcopenshell.util.element.get_container(element)
#         if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
#             return spatial_container.Name
#     except Exception:
#         pass
#     return None

# def get_material_name(element):
#     """ Extrai o nome do material associado ao elemento. """
#     material = ifcopenshell.util.element.get_material(element)
#     if not material:
#         return None
#     if hasattr(material, 'Name'):
#         return material.Name
#     elif hasattr(material, 'MaterialLayers'):
#         layer_names = [
#             layer.Material.Name 
#             for layer in material.MaterialLayers 
#             if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
#         ]
#         return ', '.join(layer_names) if layer_names else None
#     return None
    
# # --- SOLUÇÃO 2: Função de quantidade compatível com versões antigas ---
# def get_quantity_value_legacy(element, quantity_name):
#     """
#     Busca por uma quantidade específica (ex: 'Width') e retorna seu valor.
#     Esta versão é compatível com versões mais antigas do ifcopenshell sem 'get_qsets'.
#     """
#     # Itera através das relações de definição do elemento
#     for definition in getattr(element, 'IsDefinedBy', []):
#         if definition.is_a('IfcRelDefinesByProperties'):
#             prop_set = definition.RelatingPropertyDefinition
#             # Verifica se é um conjunto de quantidades (IfcElementQuantity)
#             if prop_set.is_a('IfcElementQuantity'):
#                 # Itera através das quantidades dentro do conjunto
#                 for quantity in prop_set.Quantities:
#                     if quantity.Name == quantity_name:
#                         # Extrai o valor do atributo correto (ex: LengthValue, AreaValue)
#                         value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
#                         if value_attribute:
#                             return getattr(quantity, value_attribute)
#     return None

# # --- Processamento Principal ---

# try:
#     ifc_file = ifcopenshell.open(ifc_file_path)
#     file_name = os.path.basename(ifc_file_path)

#     element_data = []
#     products = ifc_file.by_type('IfcProduct')

#     print(f"Processando {len(products)} elementos do arquivo: {file_name}...")

#     for product in products:
#         if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
#             continue

#         psets = ifcopenshell.util.element.get_psets(product)
#         rogga_pset = psets.get('PSET_RÔGGA', {})

#         element_info = {
#             'GlobalId': product.GlobalId,
#             'FileName': file_name,
#             'Class': product.is_a(),
#             'PredefinedType': getattr(product, 'PredefinedType', None),
#             'Name': getattr(product, 'Name', None),
#             'BuildingStorey': get_building_storey(product),
#             'Material': get_material_name(product),
#             'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
#             'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
            
#             # ATENÇÃO: Usando a nova função 'legacy' aqui
#             'Width': get_quantity_value_legacy(product, 'Width'),
#             'Thickness': get_quantity_value_legacy(product, 'Thickness'),
#             'Length': get_quantity_value_legacy(product, 'Length'),
#             'Height': get_quantity_value_legacy(product, 'Height'),
#         }
        
#         element_data.append(element_info)

#     df_ifc_data = pd.DataFrame(element_data)

#     desired_order = [
#         'Class', 'PredefinedType', 'BuildingStorey', 'Material', 'Name',
#         'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
#         'Width', 'Thickness', 'Length', 'Height',
#         'FileName', 'GlobalId'
#     ]

#     for col in desired_order:
#         if col not in df_ifc_data.columns:
#             df_ifc_data[col] = None
            
#     df_ifc_data = df_ifc_data[desired_order]

#     print("\nDataset criado com sucesso! Amostra dos dados:")
#     display(df_ifc_data.head().fillna(''))

# except FileNotFoundError:
#     print(f"ERRO: O arquivo não foi encontrado em: {ifc_file_path}")
# except Exception as e:
#     print(f"Ocorreu um erro inesperado: {e}")

Processando 23424 elementos do arquivo: PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00.ifc...

Dataset criado com sucesso! Amostra dos dados:


,Class,PredefinedType,BuildingStorey,Material,Name,PSET_RÔGGA.RÔGGA_SEÇÃO,PSET_RÔGGA.RÔGGA_DESCRIÇÃO,Width,Thickness,Length,Height,FileName,GlobalId
0,IfcValve,NOTDEFINED,01. TER,,"Registro de Pressão:3/4"":2734253",ø 25 x ø 25,Registro de pressão,,,,,PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00...,1n494Wk3P0mBiGCJTDJMTY
1,IfcValve,NOTDEFINED,01. TER,,"Registro de Pressão:3/4"":2736295",ø 25 x ø 25,Registro de pressão,,,,,PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00...,1n494Wk3P0mBiGCJTDJHze
2,IfcValve,NOTDEFINED,01. TER,,"Registro de Pressão:3/4"":2737404",ø 25 x ø 25,Registro de pressão,,,,,PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00...,1n494Wk3P0mBiGCJTDJHip
3,IfcValve,NOTDEFINED,01. TER,,"Registro de Pressão:3/4"":2737589",ø 25 x ø 25,Registro de pressão,,,,,PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00...,1n494Wk3P0mBiGCJTDJHfw
4,IfcValve,NOTDEFINED,01. TER,,"Registro de Pressão:3/4"":2738250",ø 25 x ø 25,Registro de pressão,,,,,PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00...,1n494Wk3P0mBiGCJTDJHU5
